In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm

# Enable cuDNN benchmark for faster GPU kernels (no effect on CPU)
torch.backends.cudnn.benchmark = True
# Use all available CPU cores for torch ops
torch.set_num_threads(os.cpu_count() or 1)




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test
        # Cache raw PIL images to avoid repeated disk I/O while keeping augmentations stochastic
        self._pil_cache = {}

    def __len__(self):
        return len(self.annotations)

    def _load_image(self, idx):
        if idx in self._pil_cache:
            return self._pil_cache[idx]
        img_path = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + ".png")
        pil_img = Image.open(img_path).convert("RGB")
        self._pil_cache[idx] = pil_img
        return pil_img

    def __getitem__(self, idx):
        pil_image = self._load_image(idx)
        if self.transform:
            image = self.transform(pil_image)
        else:
            image = pil_image
        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [3]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [4]:
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"

train_dataset = BlindnessDataset(
    train_csv_file, train_root_dir, transform=train_transform, test=False
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=os.cpu_count() // 2 or 2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=test_transform, test=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=os.cpu_count() // 2 or 2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

labels_array = train_dataset.annotations["diagnosis"].values
class_counts = np.bincount(labels_array, minlength=5)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
model_paths = {
    "resnet18": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
}
model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}

models_list = []

for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=True, num_classes=5)
    model.to(device)
    model.train()

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    num_epochs = 6  # extended epochs for better performance
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * images.size(0)

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader.dataset)
        print(
            f"Model {model_name} – Epoch [{epoch+1}/{num_epochs}] – Loss: {avg_loss:.4f}"
        )

    model.eval()
    models_list.append(model)



model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Model resnet18 – Epoch [1/6] – Loss: 1.5126
Model resnet18 – Epoch [2/6] – Loss: 1.2790
Model resnet18 – Epoch [3/6] – Loss: 1.1449
Model resnet18 – Epoch [4/6] – Loss: 1.0711
Model resnet18 – Epoch [5/6] – Loss: 1.0234
Model resnet18 – Epoch [6/6] – Loss: 0.9940


In [6]:
all_outputs = []
with torch.no_grad():
    for images in tqdm(test_loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        outputs = [model(images).unsqueeze(0) for model in models_list]
        if not outputs:
            continue
        outputs = torch.cat(outputs, dim=0)  # (n_models, batch, 5)
        averaged_outputs = torch.mean(outputs, dim=0)  # (batch, 5)
        all_outputs.extend(averaged_outputs.cpu().numpy())

all_outputs = np.array(all_outputs)
final_predictions = np.argmax(all_outputs, axis=1).astype(int)
print("Predictions shape:", final_predictions.shape)

submission_df = pd.DataFrame(
    {
        "id_code": pd.read_csv(test_csv_file)["id_code"],
        "diagnosis": final_predictions,
    }
)
submission_df.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Inference: 100%|██████████| 23/23 [00:10<00:00,  2.11it/s]

Predictions shape: (367,)
Submission saved to submission.csv
